In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/CoCare')

In [3]:
import sys, os, types, importlib

PROJECT_PATH = "/content/drive/MyDrive/CoCare"
UTILS_PATH = "/content/drive/MyDrive/CoCare/utils"

sys.path.insert(0, PROJECT_PATH)
sys.path.insert(0, UTILS_PATH)
os.chdir(PROJECT_PATH)

# Force Python to recognize utils as a package
utils_pkg = types.ModuleType("utils")
utils_pkg.__path__ = [UTILS_PATH]
sys.modules["utils"] = utils_pkg

importlib.invalidate_caches()

from utils.language_utils import detect_language
from utils.intent_utils import predict_intent
from utils.sentiment_utils import predict_sentiment
from utils.prediction_utils import predict_network_issue
from utils.response_utils import build_final_response
from utils.notification_utils import send_notification

print("✅ Imported successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Imported successfully


In [4]:
# =========================
# CoCare Chatbot Engine - FINAL READY
# =========================

from datetime import datetime
import os
import sys
import types
import glob
import importlib
import pandas as pd

# =========================
# Fix Colab Imports
# =========================
PROJECT_PATH = "/content/drive/MyDrive/CoCare"
UTILS_PATH = "/content/drive/MyDrive/CoCare/utils"

sys.path.insert(0, PROJECT_PATH)
sys.path.insert(0, UTILS_PATH)
os.chdir(PROJECT_PATH)

utils_pkg = types.ModuleType("utils")
utils_pkg.__path__ = [UTILS_PATH]
sys.modules["utils"] = utils_pkg
importlib.invalidate_caches()

from utils.language_utils import detect_language
from utils.intent_utils import predict_intent
from utils.sentiment_utils import predict_sentiment
from utils.prediction_utils import predict_network_issue
from utils.response_utils import build_final_response
from utils.notification_utils import send_notification


# =========================
# Paths
# =========================
CHAT_LOG_PATH = "/content/drive/MyDrive/CoCare/data/chat_logs.csv"

BASE_UTILS = "/content/drive/MyDrive/CoCare/utils"
NOTIFICATION_DIR = glob.glob(BASE_UTILS + "/Notifications*")[0]


def find_file(folder, clean_name):
    for filename in os.listdir(folder):
        if filename.strip() == clean_name:
            return os.path.join(folder, filename)
    raise FileNotFoundError(f"Cannot find {clean_name} inside {folder}")


INTERNAL_NOTIFICATIONS_AR_PATH = find_file(NOTIFICATION_DIR, "internal_notifications.csv")
INTERNAL_NOTIFICATIONS_EN_PATH = find_file(NOTIFICATION_DIR, "internal_notifications_en.csv")
EXTERNAL_NOTIFICATIONS_AR_PATH = find_file(NOTIFICATION_DIR, "external_notifications.csv")
EXTERNAL_NOTIFICATIONS_EN_PATH = find_file(NOTIFICATION_DIR, "external_notifications_en.csv")


# =========================
# Helpers
# =========================
def safe_int(value, default=0):
    try:
        if value == "" or pd.isna(value):
            return default
        return int(float(value))
    except Exception:
        return default


def safe_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in ["true", "1", "yes", "y"]


def get_value(row, keys, default=None):
    for key in keys:
        if key in row and row.get(key) not in ["", None]:
            return row.get(key)
    return default


def normalize_model_output(result, default_label="unknown"):
    if isinstance(result, tuple):
        label = result[0] if len(result) > 0 else default_label
        score = result[1] if len(result) > 1 else 1.0
        return label, score

    if isinstance(result, list):
        if len(result) == 0:
            return default_label, 1.0
        if isinstance(result[0], dict):
            return result[0].get("label", default_label), result[0].get("score", 1.0)
        return result[0], 1.0

    if isinstance(result, dict):
        label = result.get("label", result.get("intent", result.get("sentiment", default_label)))
        score = result.get("score", result.get("confidence", 1.0))
        return label, score

    return result, 1.0


def normalize_sentiment(sentiment):
    s = str(sentiment).lower().strip()

    if s in ["negative", "neg", "label_0", "0", "سلبي"]:
        return "negative"
    if s in ["positive", "pos", "label_2", "2", "ايجابي", "إيجابي"]:
        return "positive"
    if s in ["neutral", "neu", "label_1", "1", "محايد"]:
        return "neutral"

    return "neutral"


def normalize_intent(intent):
    s = str(intent).lower().strip()

    if s in ["complaint", "network_complaint", "technical_issue", "internet_issue"]:
        return "complaint"
    if s in ["greeting", "hello"]:
        return "greeting"
    if s in ["thanks", "thank_you"]:
        return "thanks"
    if s in ["other", "unknown", "none", ""]:
        return "unknown"

    return s


# =========================
# Load CSV Files
# =========================
def load_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path, encoding="utf-8-sig").fillna("")


def load_tables():
    return {
        "internal_ar": load_csv(INTERNAL_NOTIFICATIONS_AR_PATH),
        "internal_en": load_csv(INTERNAL_NOTIFICATIONS_EN_PATH),
        "external_ar": load_csv(EXTERNAL_NOTIFICATIONS_AR_PATH),
        "external_en": load_csv(EXTERNAL_NOTIFICATIONS_EN_PATH),
    }


def get_row(df, issue_type):
    row = df[df["issue_type"].astype(str).str.strip() == str(issue_type).strip()]

    if row.empty:
        row = df[df["issue_type"].astype(str).str.strip() == "normal"]

    if row.empty:
        raise ValueError("CSV must contain issue_type='normal' row")

    return row.iloc[0].to_dict()


# =========================
# Issue Detection
# =========================
def detect_issue_type(metrics=None, prediction=0):
    metrics = metrics or {}

    if safe_bool(metrics.get("maintenance")):
        return "maintenance"

    if safe_bool(metrics.get("outage")):
        return "outage"

    latency = metrics.get("latency")
    packet_loss = metrics.get("packet_loss")
    signal_strength = metrics.get("signal_strength")
    congestion = metrics.get("congestion")
    jitter = metrics.get("jitter")
    qos_score = metrics.get("qos_score")

    if latency is not None and float(latency) >= 250:
        return "high_latency"

    if packet_loss is not None and float(packet_loss) >= 5:
        return "packet_loss"

    if signal_strength is not None and float(signal_strength) <= -100:
        return "weak_signal"

    if congestion is not None and float(congestion) >= 80:
        return "network_congestion"

    if qos_score is not None and float(qos_score) <= 60:
        return "service_degradation"

    if jitter is not None and float(jitter) >= 50:
        return "unstable_connection"

    if prediction == 1:
        return "prediction_alert"

    return "normal"


# =========================
# Response Builders
# =========================
def build_customer_notification(issue_type, external_ar_row, external_en_row, metrics=None):
    metrics = metrics or {}

    start_time = metrics.get(
        "outage_started_at",
        datetime.now().strftime("%Y-%m-%d %H:%M")
    )

    duration = safe_int(
        metrics.get("estimated_resolution_minutes"),
        default=30
    )

    base_ar = get_value(
        external_ar_row,
        ["customer_notification_ar", "external_notification_ar", "message_ar"],
        ""
    )

    base_en = get_value(
        external_en_row,
        ["customer_notification_en", "external_notification_en", "message_en"],
        ""
    )

    message_ar = (
        f"نعتذر عن الإزعاج. {base_ar} "
        f"نوع العطل: {issue_type}. "
        f"وقت العطل: {start_time}. "
        f"المدة المتوقعة للحل: حوالي {duration} دقيقة. "
        f"نشكركم على تفهمكم."
    )

    message_en = (
        f"We apologize for the inconvenience. {base_en} "
        f"Issue type: {issue_type}. "
        f"Issue time: {start_time}. "
        f"Expected resolution time: about {duration} minutes. "
        f"Thank you for your understanding."
    )

    return message_ar, message_en


def smart_response(lang, issue_type, escalation):
    if lang == "ar":
        if escalation:
            return "نعتذر عن الإزعاج. تم تصعيد المشكلة لفريق الدعم، ونعمل حالياً على حل مشكلة الشبكة بأسرع وقت."
        return "تم رصد مشكلة في الشبكة، وسيقوم فريق الدعم بمراجعتها. نعتذر عن الإزعاج."

    if escalation:
        return "We apologize for the inconvenience. The issue has been escalated to the support team, and we are working to resolve it as soon as possible."

    return "A network issue has been detected and will be reviewed by the support team. We apologize for the inconvenience."


# =========================
# Notification Engine
# =========================
def notification_engine(intent, sentiment, prediction, history_count, metrics=None):
    metrics = metrics or {}
    tables = load_tables()

    issue_type = detect_issue_type(metrics, prediction)

    internal_ar_row = get_row(tables["internal_ar"], issue_type)
    internal_en_row = get_row(tables["internal_en"], issue_type)
    external_ar_row = get_row(tables["external_ar"], issue_type)
    external_en_row = get_row(tables["external_en"], issue_type)

    if intent in ["greeting", "thanks", "other"] or issue_type == "normal":
        return {
            "issue_type": "normal",
            "notification_type": "none",
            "escalation": False,
            "severity": "low",
            "show_to_customer": 0,
            "internal_message_ar": None,
            "internal_message_en": None,
            "external_message_ar": None,
            "external_message_en": None,
            "display_channel": "none",
            "suggested_action_ar": None,
            "suggested_action_en": None,
            "priority": 0,
            "escalate_after_attempts": 0,
            "outage_started_at": None,
            "estimated_resolution_minutes": 0,
        }

    severity = str(
        get_value(internal_en_row, ["severity"], get_value(external_en_row, ["severity"], "low"))
    ).lower()

    show_to_customer = safe_int(
        get_value(external_en_row, ["show_to_customer"], get_value(internal_ar_row, ["show_to_customer"], 1)),
        1
    )

    escalate_after = safe_int(
        get_value(internal_en_row, ["escalate_after_attempts"], get_value(internal_ar_row, ["escalate_after_attempts"], 0)),
        0
    )

    priority = safe_int(
        get_value(internal_en_row, ["priority"], get_value(internal_ar_row, ["priority"], 2)),
        2
    )

    employee_attempts = safe_int(metrics.get("employee_attempts", history_count), history_count)
    employee_resolved = metrics.get("employee_resolved", None)

    escalation = False

    if severity == "critical":
        escalation = True

    if sentiment == "negative" and history_count >= 3:
        escalation = True

    if escalate_after > 0 and employee_attempts >= escalate_after:
        escalation = True

    if employee_resolved is False:
        escalation = True

    external_message_ar, external_message_en = build_customer_notification(
        issue_type=issue_type,
        external_ar_row=external_ar_row,
        external_en_row=external_en_row,
        metrics=metrics
    )

    if escalation and show_to_customer == 1:
        notification_type = "external_noti"
        display_channel = "customer_app"
    else:
        notification_type = "internal_noti"
        display_channel = "employee_dashboard"
        external_message_ar = None
        external_message_en = None

    return {
        "issue_type": issue_type,
        "notification_type": notification_type,
        "escalation": escalation,
        "severity": severity,
        "show_to_customer": show_to_customer,
        "internal_message_ar": get_value(internal_ar_row, ["employee_notification_ar", "internal_notification_ar"]),
        "internal_message_en": get_value(internal_en_row, ["employee_notification_en", "internal_notification_en"]),
        "external_message_ar": external_message_ar,
        "external_message_en": external_message_en,
        "display_channel": display_channel,
        "suggested_action_ar": get_value(internal_ar_row, ["suggested_action_ar", "suggested_action"]),
        "suggested_action_en": get_value(internal_en_row, ["suggested_action_en", "suggested_action"]),
        "priority": priority,
        "escalate_after_attempts": escalate_after,
        "outage_started_at": metrics.get("outage_started_at"),
        "estimated_resolution_minutes": metrics.get("estimated_resolution_minutes"),
    }


# =========================
# Logging
# =========================
def log_chat(user_message, result):
    os.makedirs(os.path.dirname(CHAT_LOG_PATH), exist_ok=True)

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "user_message": user_message,
        "language": result.get("language"),
        "intent": result.get("intent"),
        "intent_confidence": result.get("intent_confidence"),
        "sentiment": result.get("sentiment"),
        "sentiment_score": result.get("sentiment_score"),
        "prediction": result.get("prediction"),
        "issue_type": result.get("issue_type"),
        "severity": result.get("severity"),
        "escalation": result.get("escalation"),
        "notification_type": result.get("notification_type"),
        "display_channel": result.get("display_channel"),
        "suggested_action_ar": result.get("suggested_action_ar"),
        "suggested_action_en": result.get("suggested_action_en"),
        "priority": result.get("priority"),
        "outage_started_at": result.get("outage_started_at"),
        "estimated_resolution_minutes": result.get("estimated_resolution_minutes"),
        "response": result.get("response"),
        "internal_message_ar": result.get("internal_message_ar"),
        "internal_message_en": result.get("internal_message_en"),
        "external_message_ar": result.get("external_message_ar"),
        "external_message_en": result.get("external_message_en"),
    }

    try:
        old_logs = pd.read_csv(CHAT_LOG_PATH, encoding="utf-8-sig")
        new_logs = pd.concat([old_logs, pd.DataFrame([row])], ignore_index=True)
    except Exception:
        new_logs = pd.DataFrame([row])

    new_logs.to_csv(CHAT_LOG_PATH, index=False, encoding="utf-8-sig")


# =========================
# Main Engine
# =========================
def process_message(user_message, metrics=None):
    metrics = metrics or {}

    lang = detect_language(user_message)

    intent_raw = predict_intent(user_message, lang)
    intent, intent_confidence = normalize_model_output(intent_raw, "unknown")
    intent = normalize_intent(intent)

    if intent in ["unknown", None, ""]:
        msg = user_message.lower()

        network_words = [
            "نت", "انترنت", "إنترنت", "شبكة", "بطيء", "بطئ",
            "ضعيف", "فصل", "يفصل", "تقطيع", "تغطية", "اشارة", "إشارة",
            "internet", "network", "slow", "weak", "disconnect",
            "disconnected", "coverage", "signal", "latency", "packet"
        ]

        if any(word in msg for word in network_words):
            intent = "complaint"
            intent_confidence = 0.90
        else:
            intent = "unknown"
            intent_confidence = 0.50

    sentiment_raw = predict_sentiment(user_message, lang)
    sentiment, sentiment_score = normalize_model_output(sentiment_raw, "neutral")
    sentiment = normalize_sentiment(sentiment)

    prediction = predict_network_issue(metrics)

    if intent_confidence < 0.60:
        decision = {
            "rule_used": "clarification",
            "prediction": prediction,
        }

        response = build_final_response(
            intent="other",
            sentiment=sentiment,
            decision=decision,
            lang=lang,
        )

        result = {
            "language": lang,
            "intent": "unknown",
            "intent_confidence": intent_confidence,
            "sentiment": sentiment,
            "sentiment_score": sentiment_score,
            "prediction": prediction,
            "issue_type": "unknown",
            "severity": None,
            "response": response,
            "escalation": False,
            "notification_type": "none",
            "display_channel": "none",
            "suggested_action_ar": None,
            "suggested_action_en": None,
            "priority": 0,
            "internal_message_ar": None,
            "internal_message_en": None,
            "external_message_ar": None,
            "external_message_en": None,
        }

        log_chat(user_message, result)
        return result

    history_count = safe_int(metrics.get("history_count", 0), 0)

    notification_decision = notification_engine(
        intent=intent,
        sentiment=sentiment,
        prediction=prediction,
        history_count=history_count,
        metrics=metrics,
    )

    decision = {
        "prediction": prediction,
        "issue_type": notification_decision["issue_type"],
        "severity": notification_decision["severity"],
        "escalation": notification_decision["escalation"],
        "notification": notification_decision["notification_type"],
        "suggested_action": notification_decision["suggested_action_en"],
        "priority": notification_decision["priority"],
    }

    if notification_decision["severity"] == "critical":
        decision["rule_used"] = "critical_network_issue"
    elif notification_decision["escalation"] and prediction == 1:
        decision["rule_used"] = "technical_high_risk"
    elif notification_decision["escalation"] and sentiment == "negative":
        decision["rule_used"] = "negative_escalation"
    elif notification_decision["issue_type"] != "normal":
        decision["rule_used"] = "network_issue_detected"
    else:
        decision["rule_used"] = "normal"

    if intent == "complaint" and notification_decision["issue_type"] != "normal":
        response = smart_response(
            lang=lang,
            issue_type=notification_decision["issue_type"],
            escalation=notification_decision["escalation"]
        )
    else:
        response = build_final_response(
            intent=intent,
            sentiment=sentiment,
            decision=decision,
            lang=lang,
        )

    result = {
        "language": lang,
        "intent": intent,
        "intent_confidence": intent_confidence,
        "sentiment": sentiment,
        "sentiment_score": sentiment_score,
        "prediction": prediction,
        "response": response,
        **notification_decision,
    }

    if result["notification_type"] != "none":
        send_notification(result["notification_type"], result)

    log_chat(user_message, result)

    return result


print("✅ CoCare Chatbot Engine FINAL loaded successfully")

✅ CoCare Chatbot Engine FINAL loaded successfully


In [9]:
metrics = {
    "latency": 320,
    "packet_loss": 2,
    "history_count": 3,
    "employee_attempts": 2,
    "employee_resolved": False,
    "outage_started_at": "2026-01-20 14:30",
    "estimated_resolution_minutes": 45
}

result = process_message("النت عندي بطيء جدا", metrics)
result

[NOTIFICATION] type=external_noti, payload={'language': 'ar', 'intent': 'complaint', 'intent_confidence': 0.9, 'sentiment': 'neutral', 'sentiment_score': 1.0, 'prediction': None, 'response': 'نعتذر عن الإزعاج. تم تصعيد المشكلة لفريق الدعم، ونعمل حالياً على حل مشكلة الشبكة بأسرع وقت.', 'issue_type': 'high_latency', 'notification_type': 'external_noti', 'escalation': True, 'severity': 'medium', 'show_to_customer': 1, 'internal_message_ar': 'تم رصد ارتفاع في زمن التأخير بالشبكة. يرجى التحقق من الضغط الحالي على الشبكة.', 'internal_message_en': 'High latency has been detected. Please check current network congestion.', 'external_message_ar': 'نعتذر عن الإزعاج. يوجد بطء أو تأخير في الشبكة حاليًا، ونعمل على معالجة المشكلة. نوع العطل: high_latency. وقت العطل: 2026-01-20 14:30. المدة المتوقعة للحل: حوالي 45 دقيقة. نشكركم على تفهمكم.', 'external_message_en': 'We apologize for the inconvenience. There is currently high latency in the network, and we are working on resolving it. Issue type: high_l

{'language': 'ar',
 'intent': 'complaint',
 'intent_confidence': 0.9,
 'sentiment': 'neutral',
 'sentiment_score': 1.0,
 'prediction': None,
 'response': 'نعتذر عن الإزعاج. تم تصعيد المشكلة لفريق الدعم، ونعمل حالياً على حل مشكلة الشبكة بأسرع وقت.',
 'issue_type': 'high_latency',
 'notification_type': 'external_noti',
 'escalation': True,
 'severity': 'medium',
 'show_to_customer': 1,
 'internal_message_ar': 'تم رصد ارتفاع في زمن التأخير بالشبكة. يرجى التحقق من الضغط الحالي على الشبكة.',
 'internal_message_en': 'High latency has been detected. Please check current network congestion.',
 'external_message_ar': 'نعتذر عن الإزعاج. يوجد بطء أو تأخير في الشبكة حاليًا، ونعمل على معالجة المشكلة. نوع العطل: high_latency. وقت العطل: 2026-01-20 14:30. المدة المتوقعة للحل: حوالي 45 دقيقة. نشكركم على تفهمكم.',
 'external_message_en': 'We apologize for the inconvenience. There is currently high latency in the network, and we are working on resolving it. Issue type: high_latency. Issue time: 2026-01-